# Обучение детектора - `yolo11n_mot20_v2`

Продакшен-детектор, общий для обоих путей (desktop и edge). Два этапа:

1. **База** - YOLO11n (COCO) дообучается на `person_dataset` → `yolo11n_people.pt`.
2. **Domain adaptation** - `yolo11n_people.pt` дообучается на кадрах MOT20 (`freeze=0`) → `yolo11n_mot20_v2.pt`.

Данные готовит `data_preparation.ipynb`

In [ ]:
import shutil
from pathlib import Path
import torch
from ultralytics import YOLO

ROOT = Path().resolve().parent
DATA_DIR = ROOT / "data"
MODELS_DIR = ROOT / "models"
RUNS_DIR = ROOT / "runs" / "detect"
MODELS_DIR.mkdir(exist_ok=True)

PERSON_YAML = DATA_DIR / "person_dataset.yaml"
MOT20_YAML = DATA_DIR / "mot20_finetune" / "dataset.yaml"

BASE_MODEL = "yolo11n.pt"
PEOPLE_WEIGHTS = MODELS_DIR / "yolo11n_people.pt"    
PROD_WEIGHTS = MODELS_DIR / "yolo11n_mot20_v2.pt"      

IMGSZ, BATCH, DEVICE, SEED = 640, 32, "cuda", 42

assert torch.cuda.is_available(), "CUDA недоступна — обучение детектора требует GPU"
torch.cuda.get_device_name(0)

In [ ]:
for y in (PERSON_YAML, MOT20_YAML):
    assert y.exists(), f"Нет датасета {y} — сначала запусти data_preparation.ipynb"
PERSON_YAML.exists(), MOT20_YAML.exists()

## Этап 1 - базовое обучение на COCO-person

In [ ]:
base = YOLO(BASE_MODEL)
r1 = base.train(
    data=str(PERSON_YAML), epochs=50, patience=20, imgsz=IMGSZ, batch=BATCH,
    device=DEVICE, seed=SEED, optimizer="auto", lr0=0.01, lrf=0.01,
    amp=True, cache="ram", fliplr=0.5, mosaic=1.0,
    project=str(RUNS_DIR), name="yolo11n_people", exist_ok=True, plots=True,
)
shutil.copy2(Path(r1.save_dir) / "weights" / "best.pt", PEOPLE_WEIGHTS)
PEOPLE_WEIGHTS

## Этап 2 - domain adaptation на MOT20 (`freeze=0`)

In [ ]:
da = YOLO(str(PEOPLE_WEIGHTS))
r2 = da.train(
    data=str(MOT20_YAML), epochs=30, imgsz=IMGSZ, batch=16,
    device=DEVICE, seed=SEED, freeze=0, lr0=0.001,
    amp=True, cache=False,
    project=str(RUNS_DIR), name="yolo11n_mot20_v2", exist_ok=True, plots=True,
)
shutil.copy2(Path(r2.save_dir) / "weights" / "best.pt", PROD_WEIGHTS)
PROD_WEIGHTS

## Валидация продакшен-детектора (MOT20 val)

Цели приёмки: Precision >= 0.90, Recall >= 0.90, F1 >= 0.88.

In [ ]:
m = YOLO(str(PROD_WEIGHTS)).val(data=str(MOT20_YAML), imgsz=IMGSZ, device=DEVICE)
p, r = float(m.box.mp), float(m.box.mr)
{"precision": round(p, 3), "recall": round(r, 3), "mAP50": round(float(m.box.map50), 3), "mAP50_95": round(float(m.box.map), 3), "F1": round(2 * p * r / (p + r + 1e-9), 3)}